In [9]:
#read the parquet file
import numpy as np
import pandas as pd

from scipy.stats import norm
from scipy.optimize import brentq



In [ ]:
df = pd.read_parquet("unpivoted_bid_offer_train.parquet", engine="pyarrow")

print(df.head())
print(df.info())
print(df.describe())
print(df.dtypes)







        date       option_ticker contract_code future_contract_code  \
0 2020-06-01  COF1C 15.00 Comdty          COF1                COF21   
1 2020-06-01  COG1C 15.00 Comdty          COG1                COG21   
2 2020-06-01  COH1C 15.00 Comdty          COH1                COH21   
3 2020-06-01  COJ1C 15.00 Comdty          COJ1                COJ21   
4 2020-06-01  COK1C 15.00 Comdty          COK1                COK21   

  option_contract_code  strike option_type expiration  settlement_price  \
0                 COF1    15.0        Call 2020-11-25             24.92   
1                 COG1    15.0        Call 2020-12-23             25.30   
2                 COH1    15.0        Call 2021-01-26             25.74   
3                 COJ1    15.0        Call 2021-02-23               NaN   
4                 COK1    15.0        Call 2021-03-26               NaN   

   settlement_vol  ... atm_strike  moneyness_usd  time_to_expiry_days  \
0             NaN  ...       40.0          -25.0 

In [ ]:
def calculate_d1_d2(f, K, sigma, T,t):
    d1 = (np.log(f / K) + 0.5*sigma ** 2 * (T-t)) / (sigma * np.sqrt(T-t))
    d2 = d1 - sigma * np.sqrt(T-t)
    print(f"d1: {d1}, d2: {d2}")
    return d1,d2


def black_76(F, K, sigma, t, T, option_type='call'):
    d1, d2 = calculate_d1_d2(F, K, sigma, t, T)

    if option_type == 'call':
        return F * norm.cdf(d1) - K * norm.cdf(d2)

    elif option_type == 'put':
        return K * norm.cdf(-d2) - F * norm.cdf(-d1)

    else:
        raise ValueError("Invalid option type")

def implied_volatility(f, K, T, t, market_price, option_type='call', tol=1e-5, max_iterations=100):
    objective_function = lambda sigma: black_76(f, K, sigma, t, T, option_type) - market_price
    return brentq(objective_function, 1e-6, 5, xtol=tol, maxiter=max_iterations)


def simulate_correlated_futures():
    return None


def price_asian_basket():
    return None